<a href="https://colab.research.google.com/github/yutaota/intro_to_causal_inference/blob/main/Ex05_IPW_and_DR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# データ生成（傾向スコアは線形、アウトカムモデルは異質性ありの非線形）

In [1]:
# 利用するライブラリのインポート
import numpy as np
import pandas as pd
import statsmodels.api as sm
import os
from sklearn.ensemble import RandomForestRegressor

# シードを設定して再現性を確保
np.random.seed(1)

# サンプル数
N = 2000

# 共変量 (X)
# 共変量は互いに独立な標準正規分布から生成
# X1:性別、X2:年齢, X3:重症度, X4:家族の経済状況, X5:遺伝的体質
# X1、X2, X3はYとAに関連し、X4, X5はYのみに関連する
X = np.random.randn(N, 5)
X[:, 0] = np.random.exponential(1, N) - 1  # 非対称、平均0付近

# 処置変数 (A): X1:性別、X2:年齢, X3:重症度に依存する。
prob_t = 1 / (1 + np.exp(-(0.5 * X[:, 0] + 0.8 * X[:, 1]+ 0.3 * X[:, 2])))
A = np.random.binomial(1, prob_t)

beta_0 = 2
delta = 1.5
u = 0.5 * X[:, 3] - 0.2 * X[:, 4] + np.random.rand(N)
Y = beta_0 * A + delta * A * X[:, 0]**2 + 1.2 * X[:, 0] + 0.6 * X[:, 1] - 0.3 * X[:, 2] + u

true_ATE = beta_0 + delta * (X[:, 0]**2).mean()
print(f"真のATE: {true_ATE:.3f}")

data = pd.DataFrame(X[:, 0:3], columns=["X1", "X2", "X3"])
data['A'] = A
data['Y'] = Y

処置人数: 1012
真のATE: 3.324


# 1. IPW推定量

In [9]:
# 傾向スコアをロジスティック回帰で推定
X_ps = sm.add_constant(data[["X1", "X2", "X3"]])
ps_model = sm.Logit(data["A"], X_ps).fit(disp=False)

e = ps_model.predict(X_ps)

# IPW による ATE 推定量
mu1_ipw = np.mean(data["A"] * data["Y"] / e)
mu0_ipw = np.mean((1 - data["A"]) * data["Y"] / (1 - e))

delta_ipw = mu1_ipw - mu0_ipw
print(f"真のATE: {true_ATE:.3f}")
print(f"IPW ATE: {delta_ipw:.3f}")

真のATE: 3.324
IPW ATE: 3.305


# 2. DR推定量（傾向スコアモデル正しい、アウトカムモデル誤り）

In [10]:
print(f"真のATE: {true_ATE:.3f}")

# アウトカムモデルを線形で推定
data1 = data[data['A'] == 1]
data0 = data[data['A'] == 0]
mu1 = sm.OLS(data1['Y'], sm.add_constant(data1[['X1', 'X2', 'X3']])).fit()
mu0 = sm.OLS(data0['Y'], sm.add_constant(data0[['X1', 'X2', 'X3']])).fit()
Y1_pred = mu1.predict(sm.add_constant(data[['X1', 'X2', 'X3']]))
Y0_pred = mu0.predict(sm.add_constant(data[['X1', 'X2', 'X3']]))
ATE_marg2_linear = (Y1_pred - Y0_pred).mean()
print(f"参考：周辺化（群別モデル-線形）: {ATE_marg2_linear:.3f}")

# 二重頑健推定
term_treated = np.mean(A * Y / e + (1 - A / e) * Y1_pred)
term_untreated = np.mean((1 - A) * Y / (1 - e) + (1 - (1 - A) / (1 - e)) * Y0_pred)
delta_dr = term_treated - term_untreated

print(f"IPW DR-1: {delta_dr:.3f}")

真のATE: 3.324
参考：周辺化（群別モデル-線形）: 3.086
IPW DR-1: 3.275


# 3. DR推定量（両モデル正しい）

In [11]:
print(f"真のATE: {true_ATE:.3f}")

# アウトカムモデルを非線形で推定
data1 = data[data['A'] == 1]
data0 = data[data['A'] == 0]
rf1 = RandomForestRegressor(n_estimators=200, random_state=1).fit(data1[Xcols], data1['Y'])
rf0 = RandomForestRegressor(n_estimators=200, random_state=1).fit(data0[Xcols], data0['Y'])
Y1_pred = rf1.predict(data[Xcols])
Y0_pred = rf0.predict(data[Xcols])
ATE_marg2_linear = (Y1_pred - Y0_pred).mean()
ATE_marg2_ml = (Y1_pred - Y0_pred).mean()
print(f"参考：周辺化（群別モデル-機械学習）: {ATE_marg2_ml:.3f}")

# 二重頑健推定
term_treated = np.mean(A * Y / e + (1 - A / e) * Y1_pred)
term_untreated = np.mean((1 - A) * Y / (1 - e) + (1 - (1 - A) / (1 - e)) * Y0_pred)
delta_dr = term_treated - term_untreated

print(f"IPW DR-2: {delta_dr:.3f}")

真のATE: 3.324
参考：周辺化（群別モデル-機械学習）: 3.330
IPW DR-2: 3.329
